In [135]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [136]:
df = pd.read_csv('quote_dataset.csv')

In [137]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [138]:
df.shape

(3038, 2)

In [139]:
df['quote'][0]

'“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”'

In [140]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


**<h3>Data Preprocessing</h3>**

*Making the text lower case and removing commas and puntuations*

In [141]:
quotes = quotes.str.lower()

In [142]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [143]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


**Tokenization**

In [144]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [145]:
vocab_size = 10000

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<unk>"
)
tokenizer.fit_on_texts(quotes)

In [146]:
word_index = tokenizer.word_index
print(len(word_index))

8979


In [147]:
sequence = tokenizer.texts_to_sequences(quotes)

In [148]:
for i in range(3):
    print(quotes[i])

“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
“it is our choices harry that show what we truly are far more than our abilities”
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”


In [149]:
for i in range(3):
    print(sequence[i])

[714, 63, 30, 20, 17, 947, 11, 8, 6, 1157, 9, 71, 294, 11, 146, 13, 810, 105, 753, 71, 2462]
[948, 8, 71, 872, 374, 10, 434, 22, 20, 466, 15, 295, 53, 55, 71, 3677]
[1338, 15, 54, 202, 715, 4, 82, 16, 37, 38, 8, 30, 330, 94, 8, 6, 1158, 2, 102, 8, 30, 330, 127, 8, 6, 3678]


**Input and output variable**

In [150]:
X = []
y = []

for seq in sequence:
    for i in range(1, len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [151]:
len(X), len(y)

(85271, 85271)

**Padding**

In [152]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [153]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [154]:
X_padded = pad_sequences(X, maxlen=max_len, padding='post')

In [155]:
y = np.array(y)

In [156]:
X_padded.shape, y.shape

((85271, 745), (85271,))

**Sparse integer labels**

In [157]:
# Keep labels as integer class IDs to avoid allocating a dense one-hot matrix.
y_one_hot = np.asarray(y, dtype=np.int32)
print(y_one_hot.shape, y_one_hot.min(), y_one_hot.max())

# Compile the model with loss="sparse_categorical_crossentropy".

(85271,) 2 8979


**Basic RNN model**

In [158]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SimpleRNN

In [159]:
embedding_dim = 50
rnn_units = 128

In [160]:
from tensorflow.keras import Input

rnn_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(vocab_size, embedding_dim, mask_zero=True),
    SimpleRNN(rnn_units),
    Dense(vocab_size, activation="softmax")
])

In [161]:
rnn_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [162]:
rnn_model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 745, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_3 (SimpleRNN)        │ (None, 128)            │        22,912 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 10000)          │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,812,912 (6.92 MB)

 Trainable params: 1,812,912 (6.92 MB)

 Non-trainable params: 0 (0.00 B)

**LSTM Model**

In [163]:
from tensorflow.keras import Input

lstm_model = Sequential([
    Input(shape=(max_len,)),
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        mask_zero=True
    ),
    LSTM(units=rnn_units),
    Dense(units=vocab_size, activation='softmax')
])

In [164]:
lstm_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [165]:
lstm_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 745, 50)        │       500,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │        91,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 10000)          │     1,290,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,881,648 (7.18 MB)

 Trainable params: 1,881,648 (7.18 MB)

 Non-trainable params: 0 (0.00 B)

**Training RNN**

In [ ]:
history_rnn = rnn_model.fit(
    X_padded,
    y_one_hot,
    validation_split=0.2,
    epochs=1,
    batch_size=128,
    verbose=1
)

**Training LSTM**

In [ ]:
history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    validation_split=0.2,
    epochs=100,
    batch_size=64
)

In [185]:
from tensorflow.keras.models import load_model
lstm_model = load_model('lstm_model.h5')

In [186]:
index_to_word = {
    index: word
    for word, index in tokenizer.word_index.items()
}

In [187]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [188]:
def predictor(model, tokenizer, text, max_len):
  text = text.lower()
  token_sequence = tokenizer.texts_to_sequences([text])

  padded_sequence = pad_sequences(
        token_sequence,
        maxlen=max_len,
        padding="post",
        truncating="pre"
    )

  predictions = model.predict(padded_sequence, verbose=0)[0]

    # Never select the padding token.
  predictions[0] = 0

  predicted_index = int(np.argmax(predictions))

  return index_to_word.get(predicted_index, "<unk>")

In [189]:
predictor(
    lstm_model,
    tokenizer,
    "the world as we",
    max_len
)

'can'

In [190]:
def generate_text(model, tokenizer, text, max_len, num_words=10):
    generated_text = text

    for _ in range(num_words):
        next_word = predictor(
            model,
            tokenizer,
            generated_text,
            max_len
        )

        if next_word == "<unk>":
            break

        generated_text += " " + next_word

    return generated_text

In [191]:
generate_text(
    lstm_model,
    tokenizer,
    "the world as we",
    max_len,
    num_words=10
)

'the world as we can only find yourself or not weep in someone come'

In [192]:
def predictor(model, tokenizer, text, max_len, temperature=0.8):
    text = text.lower()

    token_sequence = tokenizer.texts_to_sequences([text])

    padded_sequence = pad_sequences(
        token_sequence,
        maxlen=max_len,
        padding="post",
        truncating="pre"
    )

    predictions = model.predict(padded_sequence, verbose=0)[0]
    predictions[0] = 0

    # Adjust randomness using temperature.
    logits = np.log(predictions + 1e-8)
    adjusted_logits = logits / temperature
    probabilities = tf.nn.softmax(adjusted_logits).numpy()

    predicted_index = np.random.choice(
        len(probabilities),
        p=probabilities
    )

    return index_to_word.get(predicted_index, "<unk>")

In [193]:
def generate_text(
    model,
    tokenizer,
    text,
    max_len,
    num_words=10,
    temperature=1.2
):
    generated_text = text

    for _ in range(num_words):
        next_word = predictor(
            model,
            tokenizer,
            generated_text,
            max_len,
            temperature
        )

        if next_word == "<unk>":
            break

        generated_text += " " + next_word

    return generated_text

In [194]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

generate_text(
    lstm_model,
    tokenizer,
    "the world as we",
    max_len,
    num_words=10,
    temperature=0.8
)

'the world as we are where youre going to do i just want to'

In [195]:
import pickle
with open('tokenizer.pkl', 'wb') as f:
  pickle.dump(tokenizer, f)

In [196]:
with open('max_len.pkl', 'wb') as f:
  pickle.dump(max_len, f)